In [ ]:
!pip install -U youtube-transcript-api transformers accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 76.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
import re

def extract_video_id(url):
    """Extracts video ID from different YouTube URL formats."""
    # We use Regex to hunt for the 11-character ID after 'v=' or 'youtu.be/'
    match = re.search(r"(?:v=|youtu\.be/)([a-zA-Z0-9_-]{11})", url)
    return match.group(1) if match else None

def get_transcript(video_id):
    """Fetch transcript using the NEW API format."""
    try:
        api = YouTubeTranscriptApi()
        # The .fetch method grabs the subtitle object list
        transcript = api.fetch(video_id)
        # We join the list into a single long string of text
        return " ".join([t.text for t in transcript])

    except TranscriptsDisabled:
        return "Error: Transcripts are disabled for this video."
    except NoTranscriptFound:
        return "Error: No transcript found for this video."
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Check if we have a GPU (CUDA) available to speed things up
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "google/flan-t5-base"

# Load the tokenizer (translates text to numbers)
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Load the model (the neural network) and move it to the GPU/CPU
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
def summarize_chunk(text_chunk):
    # We give the model a specific instruction (prompt engineering)
    prompt = f"Summarize the following text clearly:\n{text_chunk}"

    # Convert text to tensor numbers (inputs)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    # Generate the summary
    summary_ids = model.generate(
        **inputs,
        max_new_tokens=120,    # Max length of the summary
        num_beams=4,           # Look for the 4 best paths (higher quality)
        length_penalty=1.0,    # Balance between short and long
        early_stopping=True
    )

    # Decode back to text
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [ ]:
def chunk_text(text, chunk_size=1200):
    sentences = text.split(". ")
    chunks, current_chunk = [], ""

    for sentence in sentences:
        # Check if adding the next sentence exceeds our limit
        if len(current_chunk) + len(sentence) < chunk_size:
            current_chunk += sentence + ". "
        else:
            # If full, seal the chunk and start a new one
            chunks.append(current_chunk.strip())
            current_chunk = sentence + ". "

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

In [ ]:
def generate_video_notes(video_url):
    print(f"\n🎬 Processing video: {video_url}")

    video_id = extract_video_id(video_url)
    if not video_id:
        print("Invalid YouTube URL.")
        return

    print("🎧 Fetching transcript...")
    transcript = get_transcript(video_id)

    if transcript.startswith("Error"):
        print(transcript)
        return

    print("🔪 Chunking transcript...")
    chunks = chunk_text(transcript)
    print(f"   -> {len(chunks)} chunks created.")

    print("🧠 Generating AI notes...")
    notes = []

    # Loop through chunks and summarize each one
    for i, chunk in enumerate(chunks):
        print(f"   Summarizing chunk {i+1}/{len(chunks)}...")
        summary = summarize_chunk(chunk)
        notes.append(f"- {summary}")

    print("\n" + "="*50)
    print("📝 AI GENERATED NOTES")
    print("="*50)
    print("\n".join(notes))


if __name__ == "__main__":
    url = input("Paste YouTube URL: ")
    generate_video_notes(url)

In [ ]:
!pip install streamlit

Now that Streamlit is installed, the next step is to create a Python file (`app.py`) that will contain the Streamlit application code. This file will incorporate the functions we defined earlier (`extract_video_id`, `get_transcript`, `chunk_text`, `summarize_chunk`) and build a simple web interface around them.

### Creating the Streamlit Application File (`app.py`)

This cell will create a file named `app.py` in your Colab environment. This file will encapsulate all the logic we've built, wrapped with Streamlit's UI components to make it an interactive web application.

In [ ]:
%%writefile app.py
import streamlit as st
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
import re

# --- 1. Functions from the Notebook ---

def extract_video_id(url):
    """Extracts video ID from different YouTube URL formats."""
    match = re.search(r"(?:v=|youtu\.be/)([a-zA-Z0-9_-]{11})", url)
    return match.group(1) if match else None

def get_transcript(video_id):
    """Fetch transcript using the NEW API format."""
    try:
        api = YouTubeTranscriptApi()
        transcript = api.fetch(video_id)
        return " ".join([t.text for t in transcript])
    except TranscriptsDisabled:
        return "Error: Transcripts are disabled for this video."
    except NoTranscriptFound:
        return "Error: No transcript found for this video."
    except Exception as e:
        return f"Error: {str(e)}"

def chunk_text(text, chunk_size=1200):
    sentences = text.split(". ")
    chunks, current_chunk = [], ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) < chunk_size:
            current_chunk += sentence + ". "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + ". "

    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

def summarize_chunk(text_chunk):
    # We give the model a specific instruction (prompt engineering)
    prompt = f"Summarize the following text clearly:\n{text_chunk}"

    # Convert text to tensor numbers (inputs)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    # Generate the summary
    summary_ids = model.generate(
        **inputs,
        max_new_tokens=120,
        num_beams=4,
        length_penalty=1.0,
        early_stopping=True
    )

    # Decode back to text
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# --- 2. Model Loading (cached) ---

@st.cache_resource
def load_model():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = "google/flan-t5-base"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    return tokenizer, model, device

tokenizer, model, device = load_model()

# --- 3. Streamlit UI ---

st.set_page_config(page_title="YouTube Video Summarizer", layout="wide")
st.title("🎬 YouTube Video Summarizer")
st.markdown("Enter a YouTube video URL below to get AI-generated notes.")

video_url = st.text_input("YouTube Video URL", placeholder="e.g., https://www.youtube.com/watch?v=dQw4w9WgXcQ")

if video_url:
    video_id = extract_video_id(video_url)

    if not video_id:
        st.error("Invalid YouTube URL. Please provide a valid link.")
    else:
        st.video(video_url) # Display the video
        st.subheader("Generating Notes...")

        with st.spinner("Fetching transcript..."):
            transcript = get_transcript(video_id)

        if transcript.startswith("Error"):
            st.error(transcript)
        else:
            st.success("Transcript fetched successfully!")
            st.text_area("Full Transcript (first 1000 characters)", transcript[:1000] + '...' if len(transcript) > 1000 else transcript, height=150, disabled=True)

            with st.spinner("Chunking transcript and summarizing..."):
                chunks = chunk_text(transcript)
                notes = []
                progress_bar = st.progress(0)

                for i, chunk in enumerate(chunks):
                    summary = summarize_chunk(chunk)
                    notes.append(f"- {summary}")
                    progress_bar.progress((i + 1) / len(chunks))
            st.success("Summarization complete!")

            st.subheader("📝 AI GENERATED NOTES")
            for note in notes:
                st.markdown(note)


### Running the Streamlit Application

After running the above cell to create `app.py`, you can run the Streamlit application using the following commands. Please note that the Streamlit app will run in a separate tab or window that you can access via the external URL provided by `ngrok` (if using the public URL).

1.  **Install `localtunnel` (or `ngrok`) for public access**: This is needed if you want to share the app outside of Colab.
2.  **Run Streamlit**: This will start the Streamlit server.

Execute the code cell below to run your Streamlit app:

In [ ]:
!pip install pyngrok  # Install pyngrok
!streamlit run app.py &>/dev/null& # Run Streamlit in the background

# You need to get an ngrok auth token from ngrok.com (it's free to sign up)
# Replace 'YOUR_NGROK_AUTH_TOKEN' with your actual token
!ngrok authtoken 356AcmuLVUYiehIR2JAfxbZlYbo_3XiDLbCbrxgTqhBmNQXdJ

# Start ngrok tunnel
from pyngrok import ngrok
public_url = ngrok.connect(addr='8501', proto='http')
print('Streamlit App URL:', public_url)

## Deploying to Hugging Face Spaces

To deploy your Streamlit app to Hugging Face Spaces, you'll need two main files:

1.  `app.py`: This is the Streamlit application code you've already created.
2.  `requirements.txt`: This file lists all the Python libraries your application depends on.

### 1. Create `requirements.txt`

In [ ]:
%%writefile requirements.txt
streamlit
youtube-transcript-api
transformers
accelerate
sentencepiece
torch
pyngrok

### 2. Create a New Hugging Face Space

1.  Go to [Hugging Face Spaces](https://huggingface.co/spaces).
2.  Click on "Create new Space" or your profile icon -> "New Space".
3.  Give your Space a name, choose "Streamlit" as the Space SDK, and select a license.
4.  Choose a hardware configuration (e.g., "CPU Basic").
5.  Click "Create Space".

### 3. Upload Your Files

Once your Space is created, you'll be redirected to its page. You'll see an option to upload files or connect to a Git repository.

**Using the web interface (easiest for small projects):**
1.  Click on the "Files" tab.
2.  Click "Add file" -> "Upload file".
3.  Upload your `app.py` file.
4.  Upload your `requirements.txt` file.

**Using Git (for more advanced users or larger projects):**
1.  Clone your Space's Git repository to your local machine.
2.  Copy `app.py` and `requirements.txt` into the cloned repository.
3.  Commit and push the changes to your Space.

Hugging Face Spaces will automatically detect your `app.py` and `requirements.txt` and start building and deploying your application. This might take a few minutes. Once deployed, your app will be publicly accessible at your Space's URL.